# Table 3: CPL Qwen3-8B LoRA (checkpoint-180) Memory Test

背诵 307 条刑事诉讼法 — 计算 Effective Memory / Invalid Duplicate / Error
方法：cosine similarity ≥ 0.85 (text2vec-base-chinese)，与 compare_laws.py 一致

In [1]:
# ============================================================
# Cell 1: Config & Imports
# ============================================================
import os, json, re
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Download text2vec via HF mirror
from huggingface_hub import snapshot_download
os.makedirs('/root/autodl-tmp/models', exist_ok=True)
model_dir = snapshot_download('shibing624/text2vec-base-chinese',
                               cache_dir='/root/autodl-tmp/models',
                               endpoint='https://hf-mirror.com')
from sentence_transformers import SentenceTransformer, util
sim_model = SentenceTransformer(model_dir)

# === Config ===
BASE_MODEL = '/root/autodl-tmp/employee_code/base_model'
LORA_PATH = '/root/autodl-tmp/cpl_ft/checkpoints_v2/checkpoint-180'
DATA_DIR = '/root/autodl-tmp/cpl_ft/data'
OUTPUT_DIR = '/root/autodl-tmp/cpl_ft/results_memory'
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARTICLES_FILE = os.path.join(DATA_DIR, 'criminal_procedure_articles.json')

print('PyTorch', torch.__version__)
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /root/autodl-tmp/models/models--shibing624--text2vec-base-chinese/snapshots/183bb99aa7af74355fb58d16edf8c13ae7c5433e
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PyTorch 2.8.0+cu128
CUDA: NVIDIA GeForce RTX 4080 SUPER


In [2]:
# ============================================================
# Cell 2: Load Model + LoRA checkpoint-180
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print('Loading base model...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print('Loading LoRA adapter:', LORA_PATH)
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()
print('Model ready.')

Loading base model...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading LoRA adapter: /root/autodl-tmp/cpl_ft/checkpoints_v2/checkpoint-180
Model ready.


In [3]:
# ============================================================
# Cell 3: Helpers (same as Civil Code notebook)
# ============================================================
import re

CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}
def c2i(t):
    if not t: return None
    t = str(t).strip().translate(str.maketrans('０１２３４５６７８９', '0123456789'))
    if t.isdigit(): return int(t)
    tot = sec = num = 0
    for c in t:
        if c in CN_M: num = CN_M[c]
        elif c in CN_U:
            u = CN_U[c]
            if u == 10000: sec = (sec + (num or 0)) * u; tot += sec; sec = num = 0
            else:
                if num == 0: num = 1
                sec += num * u; num = 0
        else: return None
    return tot + sec + num

def canon(a):
    if not isinstance(a, str): return ''
    x = a.strip(); x = re.sub(r'\s+', '', x); x = x.replace('（', '(').replace('）', ')')
    m = re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$', x)
    if not m: return x
    l = re.sub(r'^《中华人民共和国', '《', m.group(1))
    n = c2i(m.group(2))
    return l + '第' + str(n) + '条' if n is not None else l + '第' + m.group(2) + '条'

@torch.no_grad()
def generate(prompt, max_new=400):
    ms = [{'role': 'user', 'content': prompt}]
    tx = tokenizer.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(tx, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

PASS_THRESHOLD = 0.85
print('Helpers ready.')

Helpers ready.


In [4]:
# ============================================================
# Cell 4: Load Articles + Pre-compute Embeddings
# ============================================================
import os, json

with open(ARTICLES_FILE, 'r', encoding='utf-8') as f:
    raw_articles = json.load(f)

articles = [{'name': canon(a['name']), 'content': a['content'].strip()} for a in raw_articles]
articles = sorted(articles, key=lambda x: x['name'])
print('CPL articles:', len(articles))
print('Sample:', articles[0]['name'][:60])

standard_texts = [a['content'] for a in articles]
print('Encoding standard articles...')
standard_embeddings = sim_model.encode(standard_texts, convert_to_tensor=True, show_progress_bar=True)
print('Standard embeddings ready:', standard_embeddings.shape)

CPL articles: 307
Sample: 《刑事诉讼法》第100条
Encoding standard articles...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Standard embeddings ready: torch.Size([307, 768])


In [5]:
# ============================================================
# Cell 5: Run Memory Test (same logic as compare_laws.py)
# ============================================================
import os, json, re
from tqdm import tqdm

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'cpl_memory_checkpoint180.partial.json')
SAVE_EVERY = 100

memory_results = []
claimed_std_indices = set()
exact_count = 0
misplaced_count = 0
repetition_count = 0
fail_count = 0
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    print('Found partial save, resuming...')
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        saved = json.load(f)
    memory_results = saved['results']
    start_idx = saved['next_idx']
    exact_count = saved['exact_count']
    misplaced_count = saved['misplaced_count']
    repetition_count = saved['repetition_count']
    fail_count = saved['fail_count']
    claimed_std_indices = set(saved['claimed_std_indices'])
    print(f'  Resuming from idx={start_idx} ({start_idx}/{len(articles)} done)')

for idx in range(start_idx, len(articles)):
    art = articles[idx]
    name = art['name']
    prompt = '请背诵' + name + '。'
    raw = generate(prompt, max_new=400)

    gen_embedding = sim_model.encode(raw, convert_to_tensor=True)
    current_score = util.cos_sim(gen_embedding, standard_embeddings[idx]).item()
    cos_scores = util.cos_sim(gen_embedding, standard_embeddings)[0]
    best_idx = torch.argmax(cos_scores).item()
    best_score = cos_scores[best_idx].item()

    result = {'idx': idx, 'name': name, 'raw': raw[:200],
              'current_score': current_score, 'best_score': best_score, 'best_idx': best_idx}

    if current_score >= PASS_THRESHOLD:
        claimed_std_indices.add(idx)
        result['status_type'] = 'exact'
        exact_count += 1
    else:
        if best_score >= PASS_THRESHOLD:
            if best_idx in claimed_std_indices:
                result['status_type'] = 'repetition'
                repetition_count += 1
            else:
                result['status_type'] = 'misplaced'
                misplaced_count += 1
                claimed_std_indices.add(best_idx)
        else:
            result['status_type'] = 'fail'
            fail_count += 1

    memory_results.append(result)

    done = idx + 1
    if done % SAVE_EVERY == 0 or done == len(articles):
        valid = exact_count + misplaced_count
        print(f'  [{done}/{len(articles)}] valid={valid/done:.2%} rep={repetition_count/done:.2%} fail={fail_count/done:.2%}  -> saving...')
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({'next_idx': done, 'exact_count': exact_count, 'misplaced_count': misplaced_count,
                       'repetition_count': repetition_count, 'fail_count': fail_count,
                       'claimed_std_indices': list(claimed_std_indices), 'results': memory_results}, f, ensure_ascii=False)

total = len(articles)
valid = exact_count + misplaced_count
mem_summary = {
    'effective_memory': round(valid / total, 4),
    'invalid_duplicate': round(repetition_count / total, 4),
    'error_hallucination': round(fail_count / total, 4),
    'exact_count': exact_count, 'misplaced_count': misplaced_count,
    'repetition_count': repetition_count, 'fail_count': fail_count, 'total': total
}

print()
print('=== Memory Test Results ===')
print('Effective Memory (exact+misplaced): {:.2%} ({}/{})'.format(valid/total, valid, total))
print('  - Exact:      {:.2%} ({})'.format(exact_count/total, exact_count))
print('  - Misplaced:  {:.2%} ({})'.format(misplaced_count/total, misplaced_count))
print('Invalid Duplicate:                 {:.2%} ({})'.format(repetition_count/total, repetition_count))
print('Error/Hallucination:               {:.2%} ({})'.format(fail_count/total, fail_count))

with open(os.path.join(OUTPUT_DIR, 'cpl_memory_checkpoint180.json'), 'w', encoding='utf-8') as f:
    json.dump({'summary': mem_summary, 'per_article': memory_results}, f, ensure_ascii=False, indent=2)
print('Saved to cpl_memory_checkpoint180.json')
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

# Table 3 row
print()
print('Table 3 row:')
print('| Criminal Procedure Law | Qwen3-8B LoRA (checkpoint-180) | {:.2%} | {:.2%} | {:.2%} | 0.0969 | 0.4167 | 0.1552 |'.format(
    mem_summary['effective_memory'], mem_summary['invalid_duplicate'], mem_summary['error_hallucination']))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [100/307] valid=58.00% rep=33.00% fail=9.00%  -> saving...
  [200/307] valid=52.00% rep=40.50% fail=7.50%  -> saving...
  [300/307] valid=53.33% rep=39.00% fail=7.67%  -> saving...
  [307/307] valid=52.77% rep=39.41% fail=7.82%  -> saving...

=== Memory Test Results ===
Effective Memory (exact+misplaced): 52.77% (162/307)
  - Exact:      21.82% (67)
  - Misplaced:  30.94% (95)
Invalid Duplicate:                 39.41% (121)
Error/Hallucination:               7.82% (24)
Saved to cpl_memory_checkpoint180.json

Table 3 row:
| Criminal Procedure Law | Qwen3-8B LoRA (checkpoint-180) | 52.77% | 39.41% | 7.82% | 0.0969 | 0.4167 | 0.1552 |
